In [50]:
GPT_CONFIG_124M = {
    "vocab_size": 50257,   # Vocabulary size
    "context_length": 256, # Shortened context length (orig: 1024)
    "emb_dim": 768,        # Embedding dimension
    "n_heads": 12,         # Number of attention heads
    "n_layers": 12,        # Number of layers
    "drop_rate": 0.1,      # Dropout rate
    "qkv_bias": False      # Query-key-value bias
}

# Calculating the training and validation set losses

In [51]:
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import tiktoken
from classes import GPTModel, generate_text_simple, text_to_token_ids, token_ids_to_text

print('All imports successful!')

All imports successful!


In [52]:

class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        # Tokenize the entire text
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})

        # Use a sliding window to chunk the book into overlapping sequences of max_length
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]


def create_dataloader_v1(txt, batch_size=4, max_length=256, 
                        stride=128, shuffle=True, drop_last=True,
                        num_workers=0):

    # Initialize the tokenizer
    tokenizer = tiktoken.get_encoding("gpt2")

    # Create dataset
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    # Create dataloader
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )

    return dataloader

In [53]:
torch.manual_seed(123)  # For reproducibility
model = GPTModel(GPT_CONFIG_124M)
model.eval();  # Disable dropout during inference
tokenizer = tiktoken.get_encoding("gpt2")

We use a relatively small dataset for training the LLM (in fact, only one short story)

The reasons are:

We can run the code examples in a few minutes on a laptop computer without a suitable GPU.

The training finishes relatively fast (minutes instead of weeks), which is good for educational purposes.

We use a text from the public domain, which can be included in this GitHub repository without violating any usage rights or bloating the repository size.

For example, Llama 2 7B required 184,320 GPU hours on A100 GPUs to be trained on 2 trillion tokens

At the time of this writing, the hourly cost of an 8xA100 cloud server at AWS is approximately 30 dollars.

So, via an off-the-envelope calculation, training this LLM would cost 184,320 / 8 * 30 = 690,000 dollars

Below, we use the same dataset we used in chapter 2(The Verdict).

In [54]:
with open("data/verdict.txt", "r", encoding="utf-8") as f:
    text_data = f.read()

print(f"Text data retrieved. Length of text data: {len(text_data)} characters")

Text data retrieved. Length of text data: 20240 characters


In [55]:
# First 100 characters of the text data
print(f"First 100 characters: {text_data[:100]}")

First 100 characters: I had always thought Jack Gisburn rather a cheap genius—though a good fellow enough—so it was no gre


In [56]:
# Last 100 characters of the text data
print(f"Last 100 characters: {text_data[-100:]}")

Last 100 characters: g it for me! The Strouds stand alone, and happen once—but there’s no exterminating our kind of art.”


In [57]:
total_characters = len(text_data)
total_tokens = len(tokenizer.encode(text_data))

print(f"Total characters: {total_characters}\nTotal tokens: {total_tokens}")

Total characters: 20240
Total tokens: 5373


With 5,373 tokens, the text is very short for training an LLM(we will also load pretrained weights later).

Next, we divide the dataset into a training and a validation set and use the data loaders from chapter 2 to prepare the batches for LLM training.

For visualization purposes, the figure below assumes a max_length=6, but for the training loader, we set the max_length equal to the context length that the LLM supports.

Since we train the LLM to predict the next word in the text, the targets look the same as these inputs, except that the targets are shifted by one position

In [58]:
# Training and validation split
train_ratio = 0.90
split_idx = int(train_ratio * len(text_data))
train_data = text_data[:split_idx]
val_data = text_data[split_idx:]


train_loader = create_dataloader_v1(
    train_data,
    batch_size=2,
    max_length=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["context_length"],
    drop_last=True,
    shuffle=True,
    num_workers=0
)

val_loader = create_dataloader_v1(
    val_data,
    batch_size=2,
    max_length=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["context_length"],
    drop_last=False,
    shuffle=False,
    num_workers=0
)

print("Split data into training and validation sets completed.")

Split data into training and validation sets completed.


In [59]:
# Sanity check

if total_tokens * (train_ratio) < GPT_CONFIG_124M["context_length"]:
    print("Not enough tokens for the training loader. "
        "Try to lower the `GPT_CONFIG_124M['context_length']` or "
        "increase the `training_ratio`")

if total_tokens * (1-train_ratio) < GPT_CONFIG_124M["context_length"]:
    print("Not enough tokens for the validation loader. "
        "Try to lower the `GPT_CONFIG_124M['context_length']` or "
        "decrease the `training_ratio`")

We use a relatively small batch size to reduce the computational resource demand, and because the dataset is very small to begin with.

Llama 2 7B was trained with a batch size of 1024, for example.

An optional check that the data was loaded correctly:

In [60]:
print("Train Loader")
for x,y in train_loader:
    print(f"Input batch shape: {x.shape}, Target batch shape: {y.shape}")

print("\nValidation Loader")
for x,y in val_loader:
    print(f"Input batch shape: {x.shape}, Target batch shape: {y.shape}")

Train Loader
Input batch shape: torch.Size([2, 256]), Target batch shape: torch.Size([2, 256])
Input batch shape: torch.Size([2, 256]), Target batch shape: torch.Size([2, 256])
Input batch shape: torch.Size([2, 256]), Target batch shape: torch.Size([2, 256])
Input batch shape: torch.Size([2, 256]), Target batch shape: torch.Size([2, 256])
Input batch shape: torch.Size([2, 256]), Target batch shape: torch.Size([2, 256])
Input batch shape: torch.Size([2, 256]), Target batch shape: torch.Size([2, 256])
Input batch shape: torch.Size([2, 256]), Target batch shape: torch.Size([2, 256])
Input batch shape: torch.Size([2, 256]), Target batch shape: torch.Size([2, 256])
Input batch shape: torch.Size([2, 256]), Target batch shape: torch.Size([2, 256])

Validation Loader
Input batch shape: torch.Size([2, 256]), Target batch shape: torch.Size([2, 256])


In [61]:
train_tokens = 0
for input_batch, target_batch in train_loader:
    train_tokens += input_batch.numel()

val_tokens = 0
for input_batch, target_batch in val_loader:
    val_tokens += input_batch.numel()

print("Training tokens:", train_tokens)
print("Validation tokens:", val_tokens)
print("Ratio of training set:", train_tokens / (train_tokens + val_tokens))
print("All tokens:", train_tokens + val_tokens)

Training tokens: 4608
Validation tokens: 512
Ratio of training set: 0.9
All tokens: 5120
